In [0]:
# ===========================================
# Notebook Name:
# 01_cluster_deck_archetypes
#
# Purpose:
# Cluster functional deck compositions into
# Pokemon-based archetype candidates.
#
# Input:
# pokemon.gold.deck_similarity
# pokemon.gold.deck_registry
# pokemon.gold.deck_pokemon_features
#
# Output:
# pokemon.gold.deck_archetypes
#
# Method:
# Agglomerative hierarchical clustering
# using precomputed Weighted Jaccard distance
# ===========================================

In [0]:
%pip install scikit-learn scipy

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


DECK_SIMILARITY_TABLE = (
    "pokemon.gold.deck_similarity"
)

DECK_REGISTRY_TABLE = (
    "pokemon.gold.deck_registry"
)

DECK_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)

DECK_ARCHETYPES_TABLE = (
    "pokemon.gold.deck_archetypes"
)

print("Input :", DECK_SIMILARITY_TABLE)
print("Input :", DECK_REGISTRY_TABLE)
print("Input :", DECK_FEATURES_TABLE)
print("Output:", DECK_ARCHETYPES_TABLE)

In [0]:
registry_df = spark.table(
    DECK_REGISTRY_TABLE
)

deck_lookup_df = (
    registry_df
    .select(
        "deck_hash",
        "deck_hash_version",
        "representative_deck_code",
        "best_rank",
        "average_rank",
    )
    .dropDuplicates(["deck_hash"])
    .orderBy("deck_hash")
)

display(deck_lookup_df)

deck_count = deck_lookup_df.count()

print("Deck count:", deck_count)

In [0]:
if deck_count < 2:
    raise ValueError(
        "At least two decks are required "
        "for clustering"
    )

In [0]:
deck_lookup_pd = (
    deck_lookup_df
    .toPandas()
    .sort_values("deck_hash")
    .reset_index(drop=True)
)

deck_hashes = (
    deck_lookup_pd["deck_hash"]
    .tolist()
)

deck_index = {
    deck_hash: index
    for index, deck_hash
    in enumerate(deck_hashes)
}

display(deck_lookup_pd)

In [0]:
similarity_df = spark.table(
    DECK_SIMILARITY_TABLE
)

similarity_pd = (
    similarity_df
    .select(
        "deck_hash_a",
        "deck_hash_b",
        "weighted_jaccard_similarity",
    )
    .toPandas()
)

print(
    "Similarity pair rows:",
    len(similarity_pd),
)

In [0]:
distance_matrix = np.zeros(
    (deck_count, deck_count),
    dtype=float,
)

for row in similarity_pd.itertuples(
    index=False
):
    index_a = deck_index[
        row.deck_hash_a
    ]

    index_b = deck_index[
        row.deck_hash_b
    ]

    similarity = float(
        row.weighted_jaccard_similarity
    )

    distance = 1.0 - similarity

    distance_matrix[
        index_a,
        index_b,
    ] = distance

    distance_matrix[
        index_b,
        index_a,
    ] = distance

In [0]:
np.fill_diagonal(
    distance_matrix,
    0.0,
)

print(distance_matrix)

In [0]:
if not np.allclose(
    distance_matrix,
    distance_matrix.T,
):
    raise ValueError(
        "Distance matrix is not symmetric"
    )

if not np.allclose(
    np.diag(distance_matrix),
    0.0,
):
    raise ValueError(
        "Distance matrix diagonal is not zero"
    )

if (
    np.any(distance_matrix < 0)
    or np.any(distance_matrix > 1)
):
    raise ValueError(
        "Distance values must be "
        "between 0 and 1"
    )

print(
    "Validation passed: "
    "distance matrix is valid"
)

In [0]:
max_cluster_count = min(
    8,
    deck_count - 1,
)

cluster_evaluations = []

for cluster_count in range(
    2,
    max_cluster_count + 1,
):
    model = AgglomerativeClustering(
        n_clusters=cluster_count,
        metric="precomputed",
        linkage="average",
    )

    labels = model.fit_predict(
        distance_matrix
    )

    score = silhouette_score(
        distance_matrix,
        labels,
        metric="precomputed",
    )

    cluster_evaluations.append({
        "cluster_count": cluster_count,
        "silhouette_score": float(score),
        "labels": labels,
    })

    print(
        f"k={cluster_count}, "
        f"silhouette={score:.4f}"
    )

In [0]:
cluster_evaluation_pd = pd.DataFrame([
    {
        "cluster_count": item[
            "cluster_count"
        ],
        "silhouette_score": item[
            "silhouette_score"
        ],
    }
    for item in cluster_evaluations
])

display(cluster_evaluation_pd)

In [0]:
best_evaluation = max(
    cluster_evaluations,
    key=lambda item: item[
        "silhouette_score"
    ],
)

best_cluster_count = (
    best_evaluation[
        "cluster_count"
    ]
)

best_silhouette_score = (
    best_evaluation[
        "silhouette_score"
    ]
)

print(
    "Selected cluster count:",
    best_cluster_count,
)

print(
    "Selected silhouette score:",
    round(
        best_silhouette_score,
        4,
    ),
)

In [0]:
final_model = AgglomerativeClustering(
    n_clusters=best_cluster_count,
    metric="precomputed",
    linkage="average",
    compute_distances=True,
)

cluster_labels = (
    final_model.fit_predict(
        distance_matrix
    )
)

print(cluster_labels)

In [0]:
clustered_decks_pd = (
    deck_lookup_pd.copy()
)

clustered_decks_pd[
    "cluster_id"
] = cluster_labels.astype(int)

clustered_decks_pd[
    "model_name"
] = "agglomerative_weighted_jaccard"

clustered_decks_pd[
    "cluster_count"
] = best_cluster_count

clustered_decks_pd[
    "silhouette_score"
] = best_silhouette_score

clustered_decks_pd[
    "clustered_at"
] = datetime.now(
    timezone.utc
)

display(
    clustered_decks_pd.sort_values(
        [
            "cluster_id",
            "best_rank",
        ]
    )
)

In [0]:
cluster_size_pd = (
    clustered_decks_pd
    .groupby(
        "cluster_id",
        as_index=False,
    )
    .agg(
        cluster_size=(
            "deck_hash",
            "count",
        )
    )
)

clustered_decks_pd = (
    clustered_decks_pd
    .merge(
        cluster_size_pd,
        on="cluster_id",
        how="left",
    )
)

display(
    clustered_decks_pd.sort_values(
        [
            "cluster_id",
            "best_rank",
        ]
    )
)

In [0]:
cluster_assignment_schema = StructType([
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "cluster_id",
        IntegerType(),
        False,
    ),
])

cluster_assignment_rows = [
    {
        "deck_hash": row.deck_hash,
        "cluster_id": int(
            row.cluster_id
        ),
    }
    for row in (
        clustered_decks_pd
        .itertuples(index=False)
    )
]

cluster_assignment_df = (
    spark.createDataFrame(
        cluster_assignment_rows,
        schema=cluster_assignment_schema,
    )
)

In [0]:
cluster_pokemon_usage_df = (
    spark.table(
        DECK_FEATURES_TABLE
    )
    .join(
        cluster_assignment_df,
        on="deck_hash",
        how="inner",
    )
    .groupBy(
        "cluster_id",
        "card_name",
    )
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias(
            "decks_using_pokemon"
        ),
        F.sum(
            "quantity"
        ).alias(
            "total_copies"
        ),
        F.round(
            F.avg("quantity"),
            2,
        ).alias(
            "avg_copies_when_used"
        ),
    )
)

In [0]:
cluster_sizes_df = (
    cluster_assignment_df
    .groupBy("cluster_id")
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias(
            "cluster_size"
        )
    )
)

cluster_pokemon_usage_df = (
    cluster_pokemon_usage_df
    .join(
        cluster_sizes_df,
        on="cluster_id",
        how="left",
    )
    .withColumn(
        "cluster_inclusion_rate",
        F.col(
            "decks_using_pokemon"
        )
        / F.col("cluster_size"),
    )
    .withColumn(
        "cluster_inclusion_pct",
        F.round(
            F.col(
                "cluster_inclusion_rate"
            ) * 100,
            2,
        ),
    )
)

In [0]:
display(
    cluster_pokemon_usage_df
    .orderBy(
        "cluster_id",
        F.col(
            "cluster_inclusion_rate"
        ).desc(),
        F.col(
            "total_copies"
        ).desc(),
    )
)

In [0]:
clustered_at = datetime.now(
    timezone.utc
)

archetype_rows = []

for row in clustered_decks_pd.itertuples(
    index=False
):
    archetype_rows.append({
        "deck_hash": row.deck_hash,
        "deck_hash_version": (
            row.deck_hash_version
        ),
        "representative_deck_code": (
            row.representative_deck_code
        ),
        "cluster_id": int(
            row.cluster_id
        ),
        "cluster_size": int(
            row.cluster_size
        ),
        "model_name": row.model_name,
        "cluster_count": int(
            row.cluster_count
        ),
        "silhouette_score": float(
            row.silhouette_score
        ),
        "archetype_name": None,
        "clustered_at": clustered_at,
    })

In [0]:
archetype_schema = StructType([
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "deck_hash_version",
        StringType(),
        False,
    ),
    StructField(
        "representative_deck_code",
        StringType(),
        True,
    ),
    StructField(
        "cluster_id",
        IntegerType(),
        False,
    ),
    StructField(
        "cluster_size",
        IntegerType(),
        False,
    ),
    StructField(
        "model_name",
        StringType(),
        False,
    ),
    StructField(
        "cluster_count",
        IntegerType(),
        False,
    ),
    StructField(
        "silhouette_score",
        DoubleType(),
        True,
    ),
    StructField(
        "archetype_name",
        StringType(),
        True,
    ),
    StructField(
        "clustered_at",
        TimestampType(),
        False,
    ),
])

deck_archetypes_df = (
    spark.createDataFrame(
        archetype_rows,
        schema=archetype_schema,
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

In [0]:
duplicate_decks_df = (
    deck_archetypes_df
    .groupBy("deck_hash")
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_decks_df.count() > 0:
    display(duplicate_decks_df)

    raise ValueError(
        "Duplicate deck_hash assignments detected"
    )

In [0]:
assigned_deck_count = (
    deck_archetypes_df
    .select("deck_hash")
    .distinct()
    .count()
)

if assigned_deck_count != deck_count:
    raise ValueError(
        "Not all decks were assigned. "
        f"expected={deck_count}, "
        f"actual={assigned_deck_count}"
    )

print(
    "Validation passed: "
    "all decks assigned to one cluster"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{DECK_ARCHETYPES_TABLE}
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    representative_deck_code STRING,
    cluster_id INT NOT NULL,
    cluster_size INT NOT NULL,
    model_name STRING NOT NULL,
    cluster_count INT NOT NULL,
    silhouette_score DOUBLE,
    archetype_name STRING,
    clustered_at TIMESTAMP NOT NULL,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Machine-generated deck archetype cluster assignments'
""")

In [0]:
spark.sql(
    f"TRUNCATE TABLE "
    f"{DECK_ARCHETYPES_TABLE}"
)

(
    deck_archetypes_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        DECK_ARCHETYPES_TABLE
    )
)

print(
    "Gold deck_archetypes rebuilt."
)

In [0]:
display(
    spark.table(
        DECK_ARCHETYPES_TABLE
    )
    .orderBy(
        "cluster_id",
        "representative_deck_code",
    )
)

In [0]:
display(
    spark.table(
        DECK_ARCHETYPES_TABLE
    )
    .groupBy("cluster_id")
    .agg(
        F.count("*").alias(
            "deck_count"
        )
    )
    .orderBy("cluster_id")
)